In [ ]:
!pip install catboost

In [ ]:
!pip install optuna

In [ ]:
import pandas as pd
import numpy as np
import optuna
from sklearn.model_selection import KFold, cross_val_score
from xgboost import XGBRegressor
import warnings
warnings.filterwarnings('ignore')

train = pd.read_csv('train.txt')
test = pd.read_csv('test.txt')

train['is_train'] = 1
test['is_train'] = 0
df = pd.concat([train, test], sort=False).reset_index(drop=True)
df['data'] = pd.to_datetime(df['data'])
def engineer_features(df):
    df['rok'] = df['data'].dt.year
    df['miesiac'] = df['data'].dt.month
    df['dzien'] = df['data'].dt.day
    df['dzien_tygodnia'] = df['data'].dt.dayofweek 
    df['dzien_roku'] = df['data'].dt.dayofyear
    df['studencki_czwartek'] = (
        (df['dzien_tygodnia'] == 3) &
        (df['miesiac'].isin([10, 11, 12, 1, 3, 4, 5, 6]))
    ).astype(int)
    df['start_roku'] = ((df['miesiac'] == 10) & (df['dzien'] <= 10)).astype(int)

    df['juwenalia_czas'] = (
        (df['miesiac'] == 5) &
        (df['dzien'] > 10) &
        (df['dzien'] < 25) &
        (df['dzien_tygodnia'] >= 3)
    ).astype(int)

    df['wakacje'] = df['miesiac'].isin([7, 8, 9]).astype(int)
    df['sesja'] = df['miesiac'].isin([1, 2, 6]).astype(int)
    df['weekend'] = (df['dzien_tygodnia'] >= 5).astype(int)
    df = pd.get_dummies(df, columns=['pogoda'], prefix='pogoda', dummy_na=False)
    df['zla_pogoda_total'] = (
        df['pogoda_lekkie_opady'] +
        (df['prędkość_wiatru'] > 20).astype(int) +
        (df['temperatura'] < 0).astype(int)
    )

    return df

df = engineer_features(df)

bool_cols = ['święto', 'dzień_roboczy']
for col in bool_cols:
    df[col] = df[col].astype(int)

train_df = df[df['is_train'] == 1].copy()
test_df = df[df['is_train'] == 0].copy()

features = [c for c in train_df.columns if c not in ['id', 'data', 'studenty_ms', 'is_train', 'lockdown']]
X = train_df[features]
y_log = np.log1p(train_df['studenty_ms'])
X_test = test_df[features]

print(f"Liczba cech: {len(features)}")
print(f"Cechy: {features}")

avg_thu = train_df[train_df['studencki_czwartek'] == 1]['studenty_ms'].mean()
avg_rest = train_df[train_df['studencki_czwartek'] == 0]['studenty_ms'].mean()
print(f"\nŚrednia liczba studentów - Studenckie Czwartki: {avg_thu:.2f}")
print(f"Średnia liczba studentów - Reszta dni: {avg_rest:.2f}")
print("(Jeśli średnia w czwartki jest wyższa, nowa cecha bardzo pomoże modelowi)\n")

def objective(trial):
    param = {
        'reg_alpha': trial.suggest_float('reg_alpha', 0.001, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.01, 10.0, log=True),
        'min_child_weight': trial.suggest_int('min_child_weight', 5, 20), 
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'gamma': trial.suggest_float('gamma', 0.0, 5.0),
        'subsample': trial.suggest_float('subsample', 0.7, 0.95),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 0.95),
        'n_estimators': trial.suggest_int('n_estimators', 1500, 6000),
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.03),

        'n_jobs': -1,
        'random_state': 42,
        'tree_method': 'hist'
    }

    model = XGBRegressor(**param)

    kf = KFold(n_splits=9, shuffle=True, random_state=42)
    scores = cross_val_score(model, X, y_log, cv=kf, scoring='neg_root_mean_squared_error')

    return -scores.mean()

print("Rozpoczynam optymalizację Optuna (to potrwa chwilę)...")
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=20)

print("\nNajlepsze parametry:", study.best_params)
print("Najlepszy wynik CV (RMSLE):", study.best_value)

best_params = study.best_params
best_params['n_jobs'] = -1
best_params['random_state'] = 42
best_params['tree_method'] = 'hist'

final_model = XGBRegressor(**best_params)
final_model.fit(X, y_log)

pred_log = final_model.predict(X_test)
pred_final = np.expm1(pred_log)
pred_final = np.maximum(pred_final, 0)

sub = pd.DataFrame({'id': test_df['id'], 'studenty_ms': pred_final.round().astype(int)})
filename = 'submission_thursday_optuna.csv'
sub.to_csv(filename, index=False)

print(f"\nZapisano plik: {filename}")
print("\nStatystyki predykcji:")
print(sub['studenty_ms'].describe())

# Feature Importance Check
imp = pd.DataFrame({'Feature': features, 'Importance': final_model.feature_importances_})
print("\nTop 5 Cech (Czy studencki_czwartek jest ważny?):")
print(imp.sort_values('Importance', ascending=False).head(5))

Liczba cech: 21
Cechy: ['święto', 'dzień_roboczy', 'temperatura', 'odczuwalna_temperatura', 'wilgotność', 'prędkość_wiatru', 'rok', 'miesiac', 'dzien', 'dzien_tygodnia', 'dzien_roku', 'studencki_czwartek', 'start_roku', 'juwenalia_czas', 'wakacje', 'sesja', 'weekend', 'pogoda_lekkie_opady', 'pogoda_pochmurno', 'pogoda_ładna_pogoda', 'zla_pogoda_total']

Średnia liczba studentów - Studenckie Czwartki: 214.09
Średnia liczba studentów - Reszta dni: 48.73
(Jeśli średnia w czwartki jest wyższa, nowa cecha bardzo pomoże modelowi)

Rozpoczynam optymalizację Optuna (to potrwa chwilę)...

Najlepsze parametry: {'reg_alpha': 0.0342244932857181, 'reg_lambda': 0.6488870902501275, 'min_child_weight': 5, 'max_depth': 7, 'gamma': 3.0804496958255223, 'subsample': 0.8489039521681807, 'colsample_bytree': 0.754303683366047, 'n_estimators': 3020, 'learning_rate': 0.011458613495084494}
Najlepszy wynik CV (RMSLE): 0.6975474927395408

Zapisano plik: submission_thursday_optuna.csv

Statystyki predykcji:
count 